# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the [FAIR² dataset: Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL. The dataset follows the FAIR principles and is described by a [Croissant schema](https://mlcommons.org/croissant/).


In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.


In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL for the FAIR² dataset
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata (as an object)
print(f"Dataset: {dataset.metadata.name}\nDescription: {dataset.metadata.description}\nVersion: {dataset.metadata.version}\nIdentifier: {dataset.metadata.identifier}")

## 2. Data Overview
Explore the available record sets (`cr:RecordSet`), fields, and their `@id`s. This helps us understand the structure and what data is available.


In [ ]:
# List all record sets and their fields by @id
print("Available Record Sets:")
for record_set in dataset.record_sets:
    print(f"- Record Set @id: {record_set['@id']}")
    # Each record set contains fields (columns)
    if 'field' in record_set:
        field_ids = [f["@id"] if isinstance(f, dict) and "@id" in f else f for f in record_set["field"]]
        print(f"    Fields: {field_ids}")

## 3. Data Extraction
Load data from available record sets into pandas DataFrames for analysis. We will use the `@id` of the record set and its fields as determined above.


In [ ]:
# Gather all record set @ids
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
print(f"All record set @ids:\n{record_set_ids}\n")

# Load all record sets into DataFrames keyed by @id
dataframes = {}
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            dataframes[record_set_id] = pd.DataFrame(records)
            print(f"Loaded '{record_set_id}' with shape {dataframes[record_set_id].shape}")
        else:
            print(f"No records found for record set '{record_set_id}'")
    except Exception as e:
        print(f"Error loading '{record_set_id}': {e}")

# Show columns of a key record set for exploration
if dataframes:
    main_record_set_id = list(dataframes.keys())[0]
    print(f"\nMain record set in use: {main_record_set_id}")
    print(f"Columns: {dataframes[main_record_set_id].columns.tolist()}")
    display(dataframes[main_record_set_id].head())
else:
    print("No dataframes loaded.")

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps:
 - Filter records based on a numeric field (examples: 'cr:abundance', 'cr:coverage', 'cr:molecular_weight')
 - Normalize numeric fields
 - Group by a categorical field (e.g., 'cr:sample_name' or 'cr:accession')


In [ ]:
# Example: Choose a main record set and its fields for EDA
import numpy as np
import warnings
warnings.filterwarnings("ignore")

if dataframes:
    df = dataframes[main_record_set_id]

    # Try to locate a numeric field (try common proteomics column names)
    numeric_candidate_fields = [
        f for f in df.columns
        if pd.api.types.is_numeric_dtype(df[f]) or 'abundance' in f or 'coverage' in f or 'MW' in f or 'molecular' in f or 'peptide' in f
    ]
    if numeric_candidate_fields:
        numeric_field = numeric_candidate_fields[0]
    else:
        numeric_field = None

    print(f"Numeric field chosen: {numeric_field}")

    if numeric_field and numeric_field in df.columns:
        # We will try filtering with a threshold at the mean value
        threshold = df[numeric_field].mean()
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records in '{main_record_set_id}' with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize the selected numeric field
        filtered_df[numeric_field + "_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(f"Normalized '{numeric_field}' for filtered records:")
        display(filtered_df[[numeric_field, numeric_field + "_normalized"]].head())

        # Group by a likely categorical field (e.g. accession, sample_name, if present)
        group_candidate_fields = [f for f in df.columns if 'accession' in f or 'sample' in f or 'description' in f]
        if group_candidate_fields:
            group_field = group_candidate_fields[0]
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped data by '{group_field}':")
            display(grouped_df.head())
        else:
            print("No obvious group field found.")
    else:
        print("No numeric field found for EDA.")
else:
    print("No DataFrame to analyze.")

## 5. Visualization

Visualize distributions or relationships in the main record set. We use fields identified in the previous step.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field:
    df = dataframes[main_record_set_id]
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of '{numeric_field}' in {main_record_set_id}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # Boxplot by group if available
    if 'group_field' in locals() and group_field in df.columns:
        plt.figure(figsize=(10, 4))
        sns.boxplot(data=df, x=group_field, y=numeric_field)
        plt.title(f"'{numeric_field}' by '{group_field}'")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric field found or data not loaded for visualization.")

## 6. Conclusion

In this notebook, we demonstrated how to explore a structured mass spectrometry dataset using the `mlcroissant` library:
- The dataset definition and metadata can be loaded and inspected from a Croissant schema.
- Record sets and fields are referenced via their `@id` for consistent data handling as per the FAIR principles.
- We extracted records and performed basic exploratory analysis, including filtering, normalization, and grouping.
- Simple visualizations were produced to reveal valuable patterns in the data.

This workflow can be adapted for more advanced bioinformatics and proteomics analyses with any Croissant-formatted dataset.
